# Pipeline com Melhor Desempenho: Multinomial Naive Bayes (Lematização + N-gramas + SelectKBest k=7000)

## 1 - Montar o Google Drive

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2 - Carregar os conjuntos de dados

In [26]:
import pandas as pd

# Carregar o conjunto de dados
df_arcaico_moderno = pd.read_csv('/content/drive/MyDrive/treino/train_arcaico_moderno.csv')

# Exibir as primeiras linhas para verificar se foi carregado corretamente
print("Dataframe Arcaico x Moderno:")
display(df_arcaico_moderno.head())

Dataframe Arcaico x Moderno:


,text;style
0,És a fonte dos jardins; poço das águas vivas; ...
1,Anda com os sábios e serás sábio; mas o compan...
2,Porque eu; o Senhor; não mudo; por isso; vós; ...
3,E; ainda que nunca viu o sol; nem o conheceu; ...
4,todo o reino de Ogue; em Basã; que reinou em A...


## 3 - Pré-processamento dos dados
- Converter minúsculas e remover pontuação

In [27]:
# Separar a coluna 'text;style' em 'text' e 'target'
df_arcaico_moderno[['text', 'target']] = df_arcaico_moderno['text;style'].str.rsplit(";", n=1, expand=True)

# Limpeza básica do texto: converter para minúsculas e remover pontuação
df_arcaico_moderno['text'] = df_arcaico_moderno['text'].str.lower()
df_arcaico_moderno['text'] = df_arcaico_moderno['text'].str.replace(r'[^\w\s]', '', regex=True) # Remove pontuação

print("Dataframe Arcaico x Moderno após limpeza básica:")
display(df_arcaico_moderno.head())

Dataframe Arcaico x Moderno após limpeza básica:


,text;style,text,target
0,És a fonte dos jardins; poço das águas vivas; ...,és a fonte dos jardins poço das águas vivas qu...,arcaico
1,Anda com os sábios e serás sábio; mas o compan...,anda com os sábios e serás sábio mas o companh...,arcaico
2,Porque eu; o Senhor; não mudo; por isso; vós; ...,porque eu o senhor não mudo por isso vós ó fil...,arcaico
3,E; ainda que nunca viu o sol; nem o conheceu; ...,e ainda que nunca viu o sol nem o conheceu mai...,arcaico
4,todo o reino de Ogue; em Basã; que reinou em A...,todo o reino de ogue em basã que reinou em ast...,arcaico


- Tokenizar e remover stopwords

In [28]:
import nltk
import re
from nltk.corpus import stopwords

# Download de 'stopwords' se necessário
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Definir a lista de stopwords em português
stop_words = set(stopwords.words('portuguese'))

# Função para tokenizar usando regex e remover stopwords
def tokenize_and_remove_stopwords_regex(text):
    if isinstance(text, str):
        tokens = re.findall(r'\b[a-z]+\b', text.lower())
        filtered_tokens = [word for word in tokens if word not in stop_words]
        return filtered_tokens
    return []

# Aplicar a tokenização e remoção de stopwords
df_arcaico_moderno['text_processed'] = df_arcaico_moderno['text'].apply(tokenize_and_remove_stopwords_regex)

print("Dataframe Arcaico x Moderno após tokenização e remoção de stopwords:")
display(df_arcaico_moderno.head())

Dataframe Arcaico x Moderno após tokenização e remoção de stopwords:


,text;style,text,target,text_processed
0,És a fonte dos jardins; poço das águas vivas; ...,és a fonte dos jardins poço das águas vivas qu...,arcaico,"[fonte, jardins, vivas, correm]"
1,Anda com os sábios e serás sábio; mas o compan...,anda com os sábios e serás sábio mas o companh...,arcaico,"[anda, companheiro, tolos, afligido]"
2,Porque eu; o Senhor; não mudo; por isso; vós; ...,porque eu o senhor não mudo por isso vós ó fil...,arcaico,"[porque, senhor, mudo, filhos, sois, consumido..."
3,E; ainda que nunca viu o sol; nem o conheceu; ...,e ainda que nunca viu o sol nem o conheceu mai...,arcaico,"[ainda, nunca, viu, sol, conheceu, descanso, tal]"
4,todo o reino de Ogue; em Basã; que reinou em A...,todo o reino de ogue em basã que reinou em ast...,arcaico,"[todo, reino, ogue, reinou, astarote, edrei, f..."


- Lematização

In [29]:
import nltk
from nltk.stem import WordNetLemmatizer

# Download de 'wordnet' se necessário (usado pelo WordNetLemmatizer, com limitações para português)
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Inicializar o lematizador
lemmatizer = WordNetLemmatizer()

# Função para aplicar lematização
def apply_lemmatization(tokens):
    if isinstance(tokens, list):
        return [lemmatizer.lemmatize(word) for word in tokens]
    return []

# Aplicar lematização aos tokens processados
df_arcaico_moderno['text_lemmatized'] = df_arcaico_moderno['text_processed'].apply(apply_lemmatization)

print("Dataframe Arcaico x Moderno após lematização:")
display(df_arcaico_moderno.head())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Dataframe Arcaico x Moderno após lematização:


,text;style,text,target,text_processed,text_lemmatized
0,És a fonte dos jardins; poço das águas vivas; ...,és a fonte dos jardins poço das águas vivas qu...,arcaico,"[fonte, jardins, vivas, correm]","[fonte, jardins, viva, correm]"
1,Anda com os sábios e serás sábio; mas o compan...,anda com os sábios e serás sábio mas o companh...,arcaico,"[anda, companheiro, tolos, afligido]","[anda, companheiro, tolos, afligido]"
2,Porque eu; o Senhor; não mudo; por isso; vós; ...,porque eu o senhor não mudo por isso vós ó fil...,arcaico,"[porque, senhor, mudo, filhos, sois, consumido...","[porque, senhor, mudo, filhos, sois, consumido..."
3,E; ainda que nunca viu o sol; nem o conheceu; ...,e ainda que nunca viu o sol nem o conheceu mai...,arcaico,"[ainda, nunca, viu, sol, conheceu, descanso, tal]","[ainda, nunca, viu, sol, conheceu, descanso, tal]"
4,todo o reino de Ogue; em Basã; que reinou em A...,todo o reino de ogue em basã que reinou em ast...,arcaico,"[todo, reino, ogue, reinou, astarote, edrei, f...","[todo, reino, ogue, reinou, astarote, edrei, f..."


## 4 - Preparação e Vetorização com TF-IDF (Lematização + N-gramas)

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Juntar os tokens lematizados em strings para o TfidfVectorizer
df_arcaico_moderno['text_lemmatized_str'] = df_arcaico_moderno['text_lemmatized'].apply(lambda x: ' '.join(x))

# Inicializar o TfidfVectorizer incluindo n-gramas (unigramas, bigramas e trigramas)
tfidf_vectorizer_lemmatized_ngrams = TfidfVectorizer(max_features=9000, ngram_range=(1, 3)) # Usar max_features=9000 como na etapa anterior

# Aplicar o vetorizador ao texto lematizado
tfidf_matrix_lemmatized_ngrams = tfidf_vectorizer_lemmatized_ngrams.fit_transform(df_arcaico_moderno['text_lemmatized_str'])

print("Forma da matriz TF-IDF (Lematizada com N-gramas):", tfidf_matrix_lemmatized_ngrams.shape)

Forma da matriz TF-IDF (Lematizada com N-gramas): (36884, 9000)


## 5 - Seleção de Features com SelectKBest (k=7000)

In [31]:
from sklearn.feature_selection import SelectKBest, chi2

# Definir o número de features a selecionar
k_best_features = 7000

# Inicializar SelectKBest
selector = SelectKBest(score_func=chi2, k=k_best_features)

# Aplicar SelectKBest à matriz TF-IDF
X_kbest = selector.fit_transform(tfidf_matrix_lemmatized_ngrams, df_arcaico_moderno['target'])

print(f"Forma da matriz TF-IDF após SelectKBest (selecionando as {k_best_features} melhores features):", X_kbest.shape)

# Opcional: ver algumas das features selecionadas
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = tfidf_vectorizer_lemmatized_ngrams.get_feature_names_out()[selected_feature_indices]
print(f"\nAlgumas das {k_best_features} features selecionadas:")
print(selected_feature_names[:100])

Forma da matriz TF-IDF após SelectKBest (selecionando as 7000 melhores features): (36884, 7000)

Algumas das 7000 features selecionadas:
['abalados' 'abandonada' 'abandonadas' 'abandonado' 'abandonar'
 'abandonaram' 'abandonarei' 'abandonem' 'abandonou' 'abarim' 'abate'
 'abatido' 'abatidos' 'aberta' 'abertas' 'abertos' 'abertura'
 'abiatar sacerdote' 'abiatar sacerdotes' 'abismo' 'abismos' 'aborrece'
 'aborrecem' 'abra' 'abram' 'abre' 'abri' 'abri boca' 'abrigo' 'abrindo'
 'abrir' 'abriram' 'abriu' 'abriu porta' 'abundante' 'abundantemente'
 'acaba' 'acabando' 'acabar' 'acabaram' 'acabarei' 'acabei' 'acabou'
 'acampados' 'acampamento' 'acamparam' 'acamparamse' 'acaso' 'acazias'
 'aceita' 'aceitam' 'aceitar' 'aceitaram' 'aceitarei' 'aceite' 'aceitem'
 'aceito' 'aceitos' 'aceitou' 'acendeu' 'acendeu contra' 'acendeu ira'
 'acendeuse' 'acerca' 'aceso' 'acesso' 'acha' 'achado' 'achando' 'achar'
 'acharam' 'achareis' 'acharem' 'achava' 'ache' 'achei' 'acho' 'achou'
 'acima' 'acima todos' '

## 6 - Divisão dos dados (com features selecionadas)

In [32]:
from sklearn.model_selection import train_test_split

# Features (matriz com features selecionadas) e rótulos
X_train_kbest, X_test_kbest, y_train_kbest, y_test_kbest = train_test_split(
    X_kbest, df_arcaico_moderno['target'], test_size=0.2, random_state=42, stratify=df_arcaico_moderno['target']
)

print("Forma dos conjuntos de treino (com features selecionadas):")
print("X_train_kbest:", X_train_kbest.shape)
print("y_train_kbest:", y_train_kbest.shape)

print("\nForma dos conjuntos de teste (com features selecionadas):")
print("X_test_kbest:", X_test_kbest.shape)
print("y_test_kbest:", y_test_kbest.shape)

Forma dos conjuntos de treino (com features selecionadas):
X_train_kbest: (29507, 7000)
y_train_kbest: (29507,)

Forma dos conjuntos de teste (com features selecionadas):
X_test_kbest: (7377, 7000)
y_test_kbest: (7377,)


## 7 - Treinamento do modelo Multinomial Naive Bayes

In [33]:
from sklearn.naive_bayes import MultinomialNB

# Inicializar o modelo Multinomial Naive Bayes
model_nb_kbest = MultinomialNB()

# Treinar o modelo com os dados de treino com features selecionadas
model_nb_kbest.fit(X_train_kbest, y_train_kbest)

print("Modelo Multinomial Naive Bayes treinado com sucesso (com features selecionadas).")

Modelo Multinomial Naive Bayes treinado com sucesso (com features selecionadas).


## 8 - Avaliação do modelo

In [34]:
from sklearn.metrics import classification_report, accuracy_score

# Fazer previsões no conjunto de teste com features selecionadas
y_pred_nb_kbest = model_nb_kbest.predict(X_test_kbest)

# Avaliar o modelo
print("\nRelatório de Classificação para Arcaico x Moderno (Multinomial Naive Bayes com features selecionadas):")
print(classification_report(y_test_kbest, y_pred_nb_kbest))

accuracy_nb_kbest = accuracy_score(y_test_kbest, y_pred_nb_kbest)
print(f"Acurácia do modelo (Multinomial Naive Bayes com features selecionadas): {accuracy_nb_kbest:.4f}")


Relatório de Classificação para Arcaico x Moderno (Multinomial Naive Bayes com features selecionadas):
              precision    recall  f1-score   support

     arcaico       0.83      0.79      0.81      3689
     moderno       0.80      0.83      0.82      3688

    accuracy                           0.81      7377
   macro avg       0.81      0.81      0.81      7377
weighted avg       0.81      0.81      0.81      7377

Acurácia do modelo (Multinomial Naive Bayes com features selecionadas): 0.8137
